# UVIF v5 — Structural Uncertainty Certificate
**Outputs → Google Drive** `MyDrive/Outputs/UVIF_Bubble_Coalescence_Electrolysis_v5/`

Repeats the full certified-band analysis under three structurally distinct coverage laws:
- **S1** power law θ = k·jⁿ (Vogt–Balzer form)
- **S2** saturating full-range surrogate θ = θ_sat(1−exp(−(j/j_c)^m)) — a surrogate family of OUR construction emulating qualitative high-coverage saturation; **not** the literal Vogt (2012) relationship
- **S3** logistic surrogate θ = θ_sat/(1+(j_c/j)^m) — independent alternative family

S2 and S3 are anchored to S1 at θ = 0.25 and θ = 0.45 (two-point matching in the well-correlated regime), so all structures agree at low coverage and differ only in extrapolation — where structural risk lives.

**Certificate outputs:** optima spread across structures, intersection of certified ±10% bands, and preservation of the active mechanism. Manuscript Fig. 6 / Table 4 derive from this notebook.

**Known caveat (reported honestly):** PEM optima under S2/S3 reach the upper grid boundary (2.2 A/cm²), so the PEM spread is a lower bound.

In [ ]:
# ============================================================
# CELL 1 — GOOGLE DRIVE INITIALIZATION
# ============================================================
from pathlib import Path
import warnings
from datetime import datetime
warnings.filterwarnings("ignore")
SEED=42
IN_COLAB=False
try:
    from google.colab import drive
    IN_COLAB=True
except Exception:
    IN_COLAB=False
if IN_COLAB:
    drive.mount('/content/drive', force_remount=False)
    DRIVE_ROOT=Path('/content/drive/MyDrive')
else:
    DRIVE_ROOT=Path.cwd()
PROJECT_NAME='UVIF_Bubble_Coalescence_Electrolysis_v5'
BASE_DIR=DRIVE_ROOT/'Outputs'/PROJECT_NAME
DATA_DIR,FIG_DIR,TABLE_DIR,OUTPUT_DIR,LOG_DIR=[BASE_DIR/d for d in ['Data','Figures','Tables','Outputs','Logs']]
for f in [BASE_DIR,DATA_DIR,FIG_DIR,TABLE_DIR,OUTPUT_DIR,LOG_DIR]: f.mkdir(parents=True,exist_ok=True)
SUMMARY=OUTPUT_DIR/'outputs_summary.txt'; RUNLOG=LOG_DIR/'run_log.txt'
SUMMARY.write_text('',encoding='utf-8'); RUNLOG.write_text('',encoding='utf-8')
def log(m=''):
    stamp=datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print(f'[{stamp}] {m}')
    with open(RUNLOG,'a',encoding='utf-8') as f: f.write(f'[{stamp}] {m}\n')
    with open(SUMMARY,'a',encoding='utf-8') as f: f.write(str(m)+'\n')
log(f'IN_COLAB: {IN_COLAB}'); log(f'BASE_DIR: {BASE_DIR}')


In [ ]:
# ============================================================
# CELL 2 — IMPORTS
# ============================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.optimize import brentq
np.random.seed(SEED)
plt.rcParams.update({'font.size':8.5,'axes.grid':True,'grid.alpha':0.3,'savefig.dpi':600,
 'font.family':'serif','mathtext.fontset':'dejavuserif','legend.fontsize':7.5})
log('Libraries loaded.')


## Cell 3 — Archetype parameters, operating temperatures, budgets, grid (v3 priors)

In [ ]:
# ============================================================
# CELL 3 — PARAMETERS
# ============================================================
F=96485.0; R=8.314; T_REF=343.15
PRIORS={
 'alkaline_finite_gap':dict(j0=1e-3,b_tafel=0.060,k_theta=0.75,n_theta=0.30,k_eps=0.45,
      kappa0=0.60,gap_cm=0.30,j_lim=2.0,k_recomb=0.25,theta_crit=0.55,Ea=40e3,alpha_k=0.02),
 'alkaline_zero_gap':dict(j0=1e-3,b_tafel=0.060,k_theta=0.60,n_theta=0.30,k_eps=0.25,
      kappa0=0.60,gap_cm=0.05,j_lim=2.5,k_recomb=0.20,theta_crit=0.60,Ea=40e3,alpha_k=0.02),
 'PEM':dict(j0=1e-2,b_tafel=0.045,k_theta=0.40,n_theta=0.30,k_eps=0.10,
      kappa0=0.10,gap_cm=0.018,j_lim=6.0,k_recomb=0.10,theta_crit=0.70,Ea=40e3,alpha_k=0.02),
 'AEM':dict(j0=3e-3,b_tafel=0.055,k_theta=0.55,n_theta=0.30,k_eps=0.20,
      kappa0=0.15,gap_cm=0.005,j_lim=3.0,k_recomb=0.18,theta_crit=0.62,Ea=40e3,alpha_k=0.02),
}
T_OP={'alkaline_finite_gap':70,'alkaline_zero_gap':70,'PEM':80,'AEM':60}
BUDGETS=dict(e_min=0.62,th_max=0.50,in_max=0.20)
jg=np.linspace(0.02,2.2,441)


## Cell 4 — The three coverage-law structures (two-point anchored)

In [ ]:
# ============================================================
# CELL 4 — COVERAGE STRUCTURES
# ============================================================
def theta_S1(j,p): return np.clip(p['k_theta']*j**p['n_theta'],0,0.95)
def make_S2(p):
    # match S1 at theta=0.25 and theta=0.45 (two-point structural anchoring in the correlated regime)
    th_sat=0.95; m=1.0
    j1=(0.25/p['k_theta'])**(1/p['n_theta']); j2=(0.45/p['k_theta'])**(1/p['n_theta'])
    # solve m,jc so S2 passes through both points
    def eqs(m):
        # given m, jc from point 1; check point 2
        jc=j1/(-np.log(1-0.25/th_sat))**(1/m)
        return th_sat*(1-np.exp(-(j2/jc)**m))-0.45
    m=brentq(eqs,0.05,5.0); jc=j1/(-np.log(1-0.25/th_sat))**(1/m)
    return lambda j,p_,m=m,jc=jc,ts=th_sat: np.clip(ts*(1-np.exp(-(np.asarray(j,float)/jc)**m)),0,0.95)
def make_S3(p):
    th_sat=0.95
    j1=(0.25/p['k_theta'])**(1/p['n_theta']); j2=(0.45/p['k_theta'])**(1/p['n_theta'])
    def eqs(m):
        jc=j1*((th_sat/0.25-1))**(1/m)
        return th_sat/(1+(jc/j2)**m)-0.45
    m=brentq(eqs,0.05,8.0); jc=j1*((th_sat/0.25-1))**(1/m)
    return lambda j,p_,m=m,jc=jc,ts=th_sat: np.clip(ts/(1+(jc/np.maximum(np.asarray(j,float),1e-9))**m),0,0.95)


## Cell 5 — Electrode model parameterized by coverage structure

In [ ]:
# ============================================================
# CELL 5 — MODEL
# ============================================================
def model(j,p,T_K,theta_fn):
    j=np.asarray(j,float)
    E_rev=1.50342-9.956e-4*T_K+2.5e-7*T_K**2
    theta=theta_fn(j,p)
    j_eff=j/(1-theta)
    j0T=p['j0']*np.exp(-p['Ea']/R*(1/T_K-1/T_REF))
    eta_act=p['b_tafel']*(T_K/T_REF)*np.arcsinh(j_eff/(2*j0T))
    eps=np.clip(p['k_eps']*j,0,0.6)
    k0T=p['kappa0']*np.maximum(1+p['alpha_k']*(T_K-T_REF),0.2)
    eta_ohm=j*p['gap_cm']/(k0T*(1-eps)**1.5)
    jlim=p['j_lim']*(1-theta); frac=np.clip(j/jlim,0,0.999)
    eta_conc=(R*T_K/(2*F))*np.log(1/(1-frac))
    V=E_rev+eta_act+eta_ohm+eta_conc
    eff=1.481/V
    inst=np.clip((theta-p['theta_crit'])/(0.95-p['theta_crit']),0,1)**2+np.clip((frac-0.8)/0.2,0,1)**2
    return theta,V,eff,inst


## Cell 6 — Certified-band computation

In [ ]:
# ============================================================
# CELL 6 — CERTIFICATION
# ============================================================
def certified(p,T_C,theta_fn):
    th,V,eff,inst=model(jg,p,273.15+T_C,theta_fn)
    Pm=np.vstack([1-eff,th,inst]).T
    eps=np.array([1-BUDGETS['e_min'],BUDGETS['th_max'],BUDGETS['in_max']])
    def copt(e):
        feas=(Pm<=e).all(1)
        if not feas.any(): return None
        return int(np.max(np.where(feas)[0]))
    i=copt(eps); lo=copt(eps*0.9); hi=copt(eps*1.1)
    a=int(np.argmin(eps-Pm[i]))
    return dict(j=jg[i],lo=jg[lo if lo is not None else 0],hi=jg[hi if hi is not None else i],
                active=['energy','coverage','instability'][a])


## Cell 7 — Structural sweep: 3 structures × 4 systems

In [ ]:
# ============================================================
# CELL 7 — STRUCTURAL SWEEP
# ============================================================
rows=[]; curves={}
for s,p in PRIORS.items():
    fns={'S1 power (Vogt–Balzer form)':theta_S1,'S2 saturating surrogate':make_S2(p),'S3 logistic surrogate':make_S3(p)}
    curves[s]={}
    for lbl,fn in fns.items():
        r=certified(p,T_OP[s],fn)
        rows.append(dict(system=s,structure=lbl,j_opt=round(r['j'],4),band_lo=round(r['lo'],4),
                         band_hi=round(r['hi'],4),active=r['active']))
        curves[s][lbl]=fn(jg,p)
res=pd.DataFrame(rows)
# structural intersection band per system
inter=[]
for s in PRIORS:
    d=res[res.system==s]
    inter.append(dict(system=s,inter_lo=d.band_lo.max(),inter_hi=d.band_hi.min(),
        j_spread=round(d.j_opt.max()-d.j_opt.min(),4),
        all_coverage_active=bool((d.active=='coverage').all())))
inter=pd.DataFrame(inter)
res.to_csv(TABLE_DIR/'table_structural_bands.csv',index=False)
inter.to_csv(TABLE_DIR/'table_structural_intersection.csv',index=False)
log(res.to_string(index=False)); log(''); log(inter.to_string(index=False))


## Cell 8 — Figure R6 (manuscript Fig. 6)

In [ ]:
# ============================================================
# CELL 8 — FIGURE
# ============================================================
# figure: coverage structures + optima per system
plt.rcParams.update({'font.size':8.5,'axes.grid':True,'grid.alpha':0.3,'savefig.dpi':600,
 'font.family':'serif','mathtext.fontset':'dejavuserif','legend.fontsize':7.5})
LBL={'alkaline_finite_gap':'Alkaline, finite gap','alkaline_zero_gap':'Alkaline, zero gap','PEM':'PEM','AEM':'AEM'}
CB=dict(zip(PRIORS,['#1b6ca8','#2a9d8f','#c1121f','#7b2cbf']))
ls={'S1 power (Vogt–Balzer form)':'-','S2 saturating surrogate':'--','S3 logistic surrogate':':'}
fig,axs=plt.subplots(1,4,figsize=(7.0,2.3),sharey=True)
for ax,(s,tag) in zip(axs,zip(PRIORS,'abcd')):
    for lbl in ls:
        ax.plot(jg,curves[s][lbl],ls[lbl],color=CB[s],lw=1.5)
    ax.axhline(0.5,color='#555',lw=0.8)
    d=res[res.system==s]
    for _,r in d.iterrows():
        ax.plot(r.j_opt,0.5,'o',color=CB[s],ms=4)
    ii=inter[inter.system==s].iloc[0]
    ax.axvspan(ii.inter_lo,ii.inter_hi,color=CB[s],alpha=0.15,lw=0)
    ax.set_title(f'({tag}) {LBL[s]}\n'+rf'optima spread {d.j_opt.max()-d.j_opt.min():.3f} A cm$^{{-2}}$',loc='left',fontsize=7.5)
    ax.set_xlabel(r'$j$ (A cm$^{-2}$)')
axs[0].set_ylabel(r'$\theta(j)$')
from matplotlib.lines import Line2D
axs[3].legend(handles=[Line2D([],[],color='#444',ls=v,label=k) for k,v in ls.items()],loc='lower right',fontsize=6.2)
fig.tight_layout(); fig.savefig(FIG_DIR/'figR6_structural_uncertainty.png',dpi=600,bbox_inches='tight'); plt.close()
log('Figure saved: figR6_structural_uncertainty.png')


In [ ]:
# ============================================================
# FINAL CELL — GOOGLE DRIVE OUTPUT VERIFICATION
# ============================================================
expected=[TABLE_DIR/'table_structural_bands.csv',TABLE_DIR/'table_structural_intersection.csv',
 FIG_DIR/'figR6_structural_uncertainty.png',OUTPUT_DIR/'outputs_summary.txt',LOG_DIR/'run_log.txt']
ver=pd.DataFrame([{'file':str(p),'exists':p.exists(),'size_bytes':p.stat().st_size if p.exists() else 0} for p in expected])
ver.to_csv(OUTPUT_DIR/'google_drive_output_verification.csv',index=False)
log('Notebook v5 execution completed successfully.')
ver
